In [1]:
# 0. KHỞI TẠO MÔI TRƯỜNG LOCAL
import sys
import os
print("Đang chạy ở môi trường local máy tính.")


Đang chạy ở môi trường local máy tính.


In [2]:
import re
import unicodedata
from datasets import load_dataset
import pandas as pd

# 1. TẢI DỮ LIỆU LOCAL
print("Đang tải bộ dữ liệu...")

try:
    # Ưu tiên đọc file local từ thư mục data/raw/
    print("Thử đọc từ thư mục local data/raw/...")
    train_df = pd.read_excel("../data/raw/Train_15k.xlsx")
    val_df = pd.read_excel("../data/raw/Vadation_1,5k.xlsx")
    test_df = pd.read_excel("../data/raw/Test_1k.xlsx")
    print("Tải dữ liệu thành công từ thư mục local data/raw/!")
except FileNotFoundError:
    # Nếu không thấy file local, thử tải trực tiếp từ internet qua ID Drive làm backup
    print("Không tìm thấy file ở data/raw/. Thử tải trực tiếp qua link Google Drive...")
    file_id_train = "1AaEliMj1x9020cqkw_LxUUYgii2pUo_e"
    file_id_test = "1qiTV9iGmm0RQaZohV3mvamzuluM_4t6N"
    train_url = f"https://drive.google.com/uc?export=download&id={file_id_train}"
    test_url = f"https://drive.google.com/uc?export=download&id={file_id_test}"
    train_df = pd.read_excel(train_url)
    test_df = pd.read_excel(test_url)
    val_df = test_df.copy() # fallback
    print("Tải dữ liệu từ URL thành công!")

print(f"Train: {len(train_df):,} mẫu | Validation: {len(val_df):,} mẫu | Test: {len(test_df):,} mẫu")
print(f"Cột: {list(train_df.columns)}")
print()


c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đang tải bộ dữ liệu...
Thử đọc từ thư mục local data/raw/...
Tải dữ liệu thành công từ thư mục local data/raw/!
Train: 15,000 mẫu | Validation: 1,500 mẫu | Test: 1,000 mẫu
Cột: ['content', 'summary']



**1. TIỀN XỬ LÝ DỮ LIỆU**

1.1. Làm sạch dữ liệu

In [3]:
# 2. KHẢO SÁT SƠ BỘ

def explore(df, split_name):
    print(f"=== Khảo sát [{split_name}] ===")
    print(f"  Tổng mẫu       : {len(df):,}")
    print(f"  Null 'content' : {df['content'].isna().sum()}")
    print(f"  Null 'summary' : {df['summary'].isna().sum()}")
    print(f"  summary rỗng   : {(df['summary'].str.strip() == '').sum()}")
    print(f"  content rỗng   : {(df['content'].str.strip() == '').sum()}")
    print(f"  Độ dài content (ký tự): min={df['content'].str.len().min()}, "
          f"max={df['content'].str.len().max()}, "
          f"mean={df['content'].str.len().mean():.0f}")
    print(f"  Độ dài summary (ký tự): min={df['summary'].str.len().min()}, "
          f"max={df['summary'].str.len().max()}, "
          f"mean={df['summary'].str.len().mean():.0f}")
    print()

explore(train_df, "train")
explore(val_df,   "validation")
explore(test_df,  "test")


=== Khảo sát [train] ===
  Tổng mẫu       : 15,000
  Null 'content' : 0
  Null 'summary' : 231
  summary rỗng   : 0
  content rỗng   : 0
  Độ dài content (ký tự): min=62, max=5080, mean=1978
  Độ dài summary (ký tự): min=11.0, max=2050.0, mean=523

=== Khảo sát [validation] ===
  Tổng mẫu       : 1,500
  Null 'content' : 0
  Null 'summary' : 22
  summary rỗng   : 0
  content rỗng   : 0
  Độ dài content (ký tự): min=7, max=4745, mean=1945
  Độ dài summary (ký tự): min=7.0, max=1922.0, mean=518

=== Khảo sát [test] ===
  Tổng mẫu       : 1,000
  Null 'content' : 0
  Null 'summary' : 19
  summary rỗng   : 0
  content rỗng   : 0
  Độ dài content (ký tự): min=117, max=4670, mean=1973
  Độ dài summary (ký tự): min=61.0, max=1559.0, mean=512



In [4]:
# 3. CÁC HÀM LÀM SẠCH VĂN BẢN (Được import từ module src.preprocess)
import sys
import os
# Thêm thư mục gốc vào path để có thể import từ src
sys.path.append(os.path.abspath('..'))
from src.preprocess import clean_text, filter_df


In [5]:
# 4. ÁP DỤNG LÀM SẠCH

print("Đang làm sạch dữ liệu train...")
train_df["content"] = train_df["content"].astype(str).apply(clean_text)
train_df["summary"] = train_df["summary"].astype(str).apply(clean_text)

print("Đang làm sạch dữ liệu validation...")
val_df["content"] = val_df["content"].astype(str).apply(clean_text)
val_df["summary"] = val_df["summary"].astype(str).apply(clean_text)

print("Đang làm sạch dữ liệu test...")
test_df["content"] = test_df["content"].astype(str).apply(clean_text)
test_df["summary"] = test_df["summary"].astype(str).apply(clean_text)


Đang làm sạch dữ liệu train...
Đang làm sạch dữ liệu validation...
Đang làm sạch dữ liệu test...


In [6]:
# 5. LỌC BỎ MẪU KHÔNG HỢP LỆ

# Ngưỡng độ dài (ký tự)
MIN_CONTENT_LEN = 100    # bài báo quá ngắn → không đủ thông tin
MAX_CONTENT_LEN = 6000   # giữ phù hợp với max_length của BARTpho (~1024 token ≈ 4000-6000 ký tự)
MIN_SUMMARY_LEN = 30     # tóm tắt quá ngắn → vô nghĩa
MAX_SUMMARY_LEN = 1000   # tóm tắt dài bất thường

def filter_df(df):
    n0 = len(df)
    # Xóa null / rỗng
    df = df.dropna(subset=["content", "summary"])
    df = df[df["content"].str.strip() != ""]
    df = df[df["summary"].str.strip() != ""]

    # Lọc theo độ dài
    c_len = df["content"].str.len()
    s_len = df["summary"].str.len()
    df = df[
        (c_len >= MIN_CONTENT_LEN) & (c_len <= MAX_CONTENT_LEN) &
        (s_len >= MIN_SUMMARY_LEN) & (s_len <= MAX_SUMMARY_LEN)
    ]

    # Loại bỏ trùng lặp (content giống hệt nhau)
    df = df.drop_duplicates(subset=["content"])

    # Loại bỏ mẫu có summary dài hơn / bằng content (tóm tắt không hợp lý)
    df = df[df["summary"].str.len() < df["content"].str.len()]

    print(f"  Trước lọc: {n0:,} | Sau lọc: {len(df):,} "
          f"| Đã loại: {n0 - len(df):,} ({(n0-len(df))/n0*100:.1f}%)")
    return df.reset_index(drop=True)

print("\nLọc dữ liệu train...")
train_clean = filter_df(train_df)
print("Lọc dữ liệu validation...")
val_clean   = filter_df(val_df)
print("Lọc dữ liệu test...")
test_clean  = filter_df(test_df)



Lọc dữ liệu train...
  Trước lọc: 15,000 | Sau lọc: 14,129 | Đã loại: 871 (5.8%)
Lọc dữ liệu validation...
  Trước lọc: 1,500 | Sau lọc: 1,422 | Đã loại: 78 (5.2%)
Lọc dữ liệu test...
  Trước lọc: 1,000 | Sau lọc: 951 | Đã loại: 49 (4.9%)


In [7]:
# 6. THỐNG KÊ SAU LÀM SẠCH

print("\n=== Thống kê sau làm sạch ===")
explore(train_clean, "train_clean")
explore(val_clean,   "val_clean")
explore(test_clean,  "test_clean")



=== Thống kê sau làm sạch ===
=== Khảo sát [train_clean] ===
  Tổng mẫu       : 14,129
  Null 'content' : 0
  Null 'summary' : 0
  summary rỗng   : 0
  content rỗng   : 0
  Độ dài content (ký tự): min=106, max=5075, mean=1904
  Độ dài summary (ký tự): min=34, max=999, mean=501

=== Khảo sát [val_clean] ===
  Tổng mẫu       : 1,422
  Null 'content' : 0
  Null 'summary' : 0
  summary rỗng   : 0
  content rỗng   : 0
  Độ dài content (ký tự): min=142, max=4697, mean=1874
  Độ dài summary (ký tự): min=74, max=979, mean=496

=== Khảo sát [test_clean] ===
  Tổng mẫu       : 951
  Null 'content' : 0
  Null 'summary' : 0
  summary rỗng   : 0
  content rỗng   : 0
  Độ dài content (ký tự): min=116, max=4404, mean=1888
  Độ dài summary (ký tự): min=67, max=995, mean=500



In [8]:
# 7. LƯU KẾT QUẢ
import os
os.makedirs("../data/processed", exist_ok=True)
train_clean.to_excel("../data/processed/train_clean.xlsx", index=False)
val_clean.to_excel("../data/processed/val_clean.xlsx", index=False)
test_clean.to_excel("../data/processed/test_clean.xlsx", index=False)
print("Đã lưu kết quả vào thư mục: ../data/processed/train_clean.xlsx, ../data/processed/val_clean.xlsx và ../data/processed/test_clean.xlsx")


Đã lưu kết quả vào thư mục: ../data/processed/train_clean.xlsx, ../data/processed/val_clean.xlsx và ../data/processed/test_clean.xlsx


In [9]:
from transformers import AutoTokenizer
from datasets import Dataset
import sys
import os
sys.path.append(os.path.abspath('..'))
from src.model import preprocess_tokenize_function

# Load tokenizer BARTPho
model_checkpoint = "vinai/bartpho-word"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Chuyển DataFrame sang Dataset
train_dataset = Dataset.from_pandas(train_clean)
val_dataset = Dataset.from_pandas(val_clean)
test_dataset = Dataset.from_pandas(test_clean)

print("Đang chuyển text thành token IDs...")
tokenized_train = train_dataset.map(
    lambda x: preprocess_tokenize_function(x, tokenizer, max_input_length=1024, max_target_length=256),
    batched=True
)
tokenized_val = val_dataset.map(
    lambda x: preprocess_tokenize_function(x, tokenizer, max_input_length=1024, max_target_length=256),
    batched=True
)
tokenized_test = test_dataset.map(
    lambda x: preprocess_tokenize_function(x, tokenizer, max_input_length=1024, max_target_length=256),
    batched=True
)
print("Hoàn tất chuyển text -> token IDs!")


Đang chuyển text thành token IDs...


Map: 100%|██████████| 951/951 [00:16<00:00, 57.15 examples/s]

Hoàn tất chuyển text -> token IDs!


In [10]:
# Hiển thị văn bản gốc
print("===== TEXT GỐC =====")
print(train_clean["content"][0])

# Hiển thị token IDs
print("\n===== TOKEN IDs =====")
print(tokenized_train[0]["input_ids"])

# Hiển thị token tương ứng
print("\n===== TOKENS =====")
tokens = tokenizer.convert_ids_to_tokens(
    tokenized_train[0]["input_ids"]
)
print(tokens)

===== TEXT GỐC =====
Thủ tướng Nguyễn Xuân Phúc và Bộ trưởng Ngoại giao Nhật Bản Taro Kono -
Tại buổi tiếp, Thủ tướng Nguyễn Xuân Phúc bày tỏ vui mừng trước sự phát triển nhanh chóng, thực chất và toàn diện của quan hệ Đối tác chiến lược sâu rộng Việt Nam - Nhật Bản; cho rằng việc hai nước đẩy mạnh trao đổi và tiếp xúc đoàn cấp cao thời gian qua đã giúp tăng cường sự tin cậy- điều kiện quan trọng để thúc đẩy quan hệ Việt Nam - Nhật Bản phát triển hơn nữa trên mọi lĩnh vực; mong muốn tăng cường hợp tác ASEAN - Nhật Bản và Mekong - Nhật Bản, nhất là khi Việt Nam đảm nhiệm vai trò Điều phối viên quan hệ ASEAN - Nhật Bản giai đoạn 2018 - 2021; đánh giá cao Nhật Bản đã cung cấp vốn hỗ trợ phát triển chính thức ODA cho Việt Nam thời gian qua, khẳng định Chính phủ Việt Nam coi trọng và nỗ lực thực hiện hiệu quả các dự án vốn vay ODA giữa hai nước.
Thủ tướng Nguyễn Xuân Phúc đề nghị Bộ trưởng Ngoại giao Nhật Bản quan tâm thúc đẩy liên kết kinh tế giữa 2 nước, đặc biệt là những lĩnh vực có nhiề

1.4. Giới hạn đồ dài bằng padding & truncation

In [11]:
from src.model import preprocess_tokenize_function

print("Đang thực hiện padding & truncation...")

# Áp dụng cho train set
processed_train = train_dataset.map(
    lambda x: preprocess_tokenize_function(x, tokenizer, max_input_length=1024, max_target_length=256),
    batched=True
)

# Áp dụng cho validation set
processed_val = val_dataset.map(
    lambda x: preprocess_tokenize_function(x, tokenizer, max_input_length=1024, max_target_length=256),
    batched=True
)

# Áp dụng cho test set
processed_test = test_dataset.map(
    lambda x: preprocess_tokenize_function(x, tokenizer, max_input_length=1024, max_target_length=256),
    batched=True
)

print("Hoàn tất padding & truncation!")
print("\nĐỘ DÀI INPUT: ")
print(len(processed_train[0]["input_ids"]))
print("\n:ĐỘ DÀI LABEL:")
print(len(processed_train[0]["labels"]))


Đang thực hiện padding & truncation...


Map: 100%|██████████| 951/951 [00:12<00:00, 73.48 examples/s]

Hoàn tất padding & truncation!

ĐỘ DÀI INPUT: 
1024

:ĐỘ DÀI LABEL:
256


In [12]:
print(tokenizer.pad_token)
print(tokenizer.pad_token_id)


<pad>
1


In [13]:
# Hiển thị input_ids & attetion_mask cửa mẫu đầu trong tập train
print("\n========== TRAIN SAMPLE ==========")
print(processed_train[0]["input_ids"])
print(processed_train[0]["attention_mask"])
# Hiển thị input_ids & attetion_mask cửa mẫu đầu trong tập test
print("\n========== TEST SAMPLE ==========")
print(processed_test[0]["input_ids"])
print(processed_test[0]["attention_mask"])


========== TRAIN SAMPLE ==========
[0, 11639, 3110, 3751, 2334, 2777, 6, 125, 676, 14367, 574, 1219, 3156, 8197, 1595, 47381, 770, 3, 382, 391, 17525, 1487, 4, 11639, 3110, 3751, 2334, 2777, 5744, 1405, 911, 2766, 71, 61, 1073, 12155, 485, 47108, 1701, 4, 2927, 567, 6, 668, 1283, 7, 2665, 2301, 33784, 18116, 2856, 8536, 808, 720, 350, 590, 31, 1219, 20405, 1301, 65, 13, 87, 49, 82, 58, 1172, 384, 906, 1361, 6, 917, 11796, 639, 181, 84, 790, 4365, 89, 14, 171, 128, 13458, 61, 200, 33430, 1581, 31, 184, 2379, 2665, 7594, 24, 15022, 1172, 2665, 2301, 350, 590, 31, 1219, 3156, 1073, 12155, 48, 348, 34, 207, 7362, 1659, 18415, 1395, 65, 1790, 202, 128, 13458, 2288, 18116, 1499, 31, 1219, 3156, 6, 10225, 31, 1219, 20405, 1301, 4, 67, 8, 26, 350, 590, 15758, 9393, 759, 2680, 432, 5438, 1430, 2665, 2301, 1499, 31, 1219, 3156, 23720, 687, 501, 31, 3914, 8867, 65, 480, 133, 84, 1219, 3156, 14, 1820, 181, 244, 43451, 18075, 1073, 12155, 159, 2908, 6803, 13, 350, 590, 790, 4365, 38937, 4, 50351, 